# Solar Aruba: analisis exploratorio de la energia estimada
**Grupo 4 - Curso 2026/27**

Este cuaderno estudia `energia_estimada_kwh` como **variable objetivo** a partir de los datos
meteorologicos, solares y energeticos calculados para cada estacion e instante de cinco minutos.
La energia estimada procede de un escenario con potencia nominal de 7,5 MW y factor de suciedad
supuesto; no es generacion electrica medida.

Preguntas principales:
1. Hay datos ausentes, duplicados o valores de energia que debamos revisar?
2. Como cambia la energia estimada por hora, mes y estacion?
3. Que relacion tiene con la meteorologia y las variables solares?
4. Que informacion historica puede usarse para predecir la energia a 15 minutos?

**Para ejecutarlo:** guardar el notebook junto al CSV y ejecutar las celdas de arriba abajo.

## 1. Carga y primera revisión
Empezamos por cargar el archivo y revisar las columnas. Conservamos una copia de los datos
originales para poder comparar después de la preparación.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATASET_PATH = DATA_DIR / "aruba_cruzado_5min_con_energia.csv"

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timezone, timedelta
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 35)

datos_originales = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
datos = datos_originales.copy()
objetivo = "energia_estimada_kwh"

print("Filas:", datos.shape[0])
print("Columnas:", datos.shape[1])
display(datos.head())


In [ ]:
datos.info()


### Que representa cada grupo de variables
Cada fila corresponde a una estacion y un instante de cinco minutos. El archivo mezcla lecturas
meteorologicas con datos solares y variables energeticas calculadas.

| Grupo | Columnas | Uso en el analisis |
|---|---|---|
| Estacion | `station_id`, `station_name`, `station_code`, `district` | Identificar estaciones |
| Ubicacion | `latitude`, `longitude`, `elevation_m` | Situar las estaciones |
| Control | `is_active`, `reading_id`, `reading_available` | Revisar disponibilidad |
| Tiempo | `timestamp` | Ordenar observaciones y construir retardos |
| Meteorologia | `temperature_c`, `humidity_pct`, `wind_speed_kmh`, `wind_direction_deg`, `pressure_hpa`, `precipitation_mm`, `visibility_km`, `uv_index` | Explicar la energia estimada |
| Radiacion | `shortwave_radiation_instant`, `direct_normal_irradiance_instant`, `diffuse_radiation_instant`, `direct_radiation_instant` | Variables solares explicativas |
| Condiciones solares | `cloud_cover`, `is_day`, `sunshine_duration_source_s` | Nubosidad y dia/noche |
| Energia estimada | `potencia_estimada_mw`, `energia_estimada_kwh`, `soiling_factor_supuesto` | Objetivo y escenario de calculo |
| Origen del cruce | `radiation_source_timestamp`, `radiation_quality` | Procedencia temporal |

La energia estimada es calculada, por lo que sus relaciones con las variables que participan en
la formula no deben interpretarse como mediciones independientes.


In [ ]:
# Las fechas del CSV llevan Z: se interpretan como UTC.
datos["timestamp"] = pd.to_datetime(datos["timestamp"], utc=True, errors="coerce")
datos["radiation_source_timestamp"] = pd.to_datetime(
    datos["radiation_source_timestamp"], utc=True, errors="coerce"
)

# Para los gráficos horarios usamos la hora de Aruba (UTC-4).
zona_aruba = timezone(timedelta(hours=-4))
datos["fecha_local"] = datos["timestamp"].dt.tz_convert(zona_aruba)
datos["hora"] = datos["fecha_local"].dt.hour
datos["mes"] = datos["fecha_local"].dt.strftime("%Y-%m")

display(datos.groupby("station_name").agg(
    filas=("timestamp", "size"),
    inicio_utc=("timestamp", "min"),
    fin_utc=("timestamp", "max"),
    lecturas_disponibles=("reading_available", "sum")
))
display(datos[["station_name", "station_code", "district", "latitude", "longitude", "elevation_m"]]
        .drop_duplicates().reset_index(drop=True))


El archivo tiene **219.984 filas y 28 columnas**, repartidas entre San Nicolaas y Queen Beatrix
Airport. En UTC cubre del 1 de septiembre de 2025 al 17 de septiembre de 2026. Hay algo más de
un año de datos, por lo que podremos describir diferencias mensuales, pero no comprobar que
se repitan durante varios años.


## 2. Calidad de los datos
Revisamos primero los ausentes y después los duplicados, la frecuencia temporal y los rangos.
Una fila creada en el cruce puede tener radiación aunque no exista lectura meteorológica.


In [ ]:
faltantes = pd.DataFrame({
    "ausentes": datos_originales.isna().sum(),
    "porcentaje": datos_originales.isna().mean() * 100
}).sort_values("ausentes", ascending=False)
display(faltantes.round(2))

faltantes[faltantes["ausentes"] > 0]["porcentaje"].plot.barh(color="steelblue")
plt.title("Porcentaje de datos ausentes por variable")
plt.xlabel("Datos ausentes (%)")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

display(datos["reading_available"].value_counts(dropna=False).rename("filas").to_frame())
display(pd.crosstab(datos["reading_available"], datos["temperature_c"].isna(),
                    colnames=["Temperatura ausente"]))


In [ ]:
# El porcentaje diario permite ver si los ausentes se concentran en periodos completos.
disponibilidad = datos.groupby(["station_name", pd.Grouper(key="timestamp", freq="D")])[
    "reading_available"
].mean().unstack("station_name") * 100

disponibilidad.plot(linewidth=1)
plt.title("Disponibilidad diaria de lecturas meteorológicas por estación")
plt.xlabel("Fecha (UTC)")
plt.ylabel("Lecturas disponibles (%)")
plt.ylim(-5, 105)
plt.legend(title="Estación")
plt.tight_layout()
plt.show()

print("Filas originales duplicadas:", datos_originales.duplicated().sum())
print("Claves estación-fecha duplicadas:", datos.duplicated(["station_id", "timestamp"]).sum())
print("Fechas no válidas:", datos["timestamp"].isna().sum())
print("Fechas de origen no válidas:", datos["radiation_source_timestamp"].isna().sum())

datos = datos.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
intervalos = datos.groupby("station_id")["timestamp"].diff()
display(intervalos.value_counts().rename("intervalos").to_frame())


Hay **8.640 filas sin lectura meteorologica (3,93 %)**. La energia estimada tambien queda
sin valor en esas filas porque su calculo depende de variables meteorologicas. No rellenamos
huecos de varios dias ni cambiamos ausentes por ceros.

**Decision:** en cada relacion se descartan unicamente las filas ausentes necesarias. La
interpretacion distingue entre energia estimada y generacion electrica medida.

In [ ]:
meteorologicas = ["temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
                  "pressure_hpa", "precipitation_mm", "visibility_km", "uv_index"]
solares = [objetivo, "direct_normal_irradiance_instant", "diffuse_radiation_instant",
           "direct_radiation_instant", "cloud_cover", "sunshine_duration_source_s"]

display(datos[meteorologicas + solares].describe().T.round(2))

rangos = pd.Series({
    "Humedad fuera de 0-100 %": ((datos["humidity_pct"] < 0) | (datos["humidity_pct"] > 100)).sum(),
    "Nubosidad fuera de 0-100 %": ((datos["cloud_cover"] < 0) | (datos["cloud_cover"] > 100)).sum(),
    "Dirección fuera de 0-360 grados": ((datos["wind_direction_deg"] < 0) | (datos["wind_direction_deg"] > 360)).sum(),
    "Velocidad del viento negativa": (datos["wind_speed_kmh"] < 0).sum(),
    "Precipitación negativa": (datos["precipitation_mm"] < 0).sum(),
    "Energía estimada negativa": (datos[objetivo] < 0).sum(),
    "Radiación positiva con is_day=0": ((datos["is_day"] == 0) & (datos[objetivo] > 0)).sum(),
    "Radiación cero con is_day=1": ((datos["is_day"] == 1) & (datos[objetivo] == 0)).sum()
}, name="filas")
display(rangos.to_frame())

# El criterio IQR sirve para señalar valores extremos, no para borrarlos automáticamente.
q1 = datos[meteorologicas].quantile(0.25)
q3 = datos[meteorologicas].quantile(0.75)
iqr = q3 - q1
extremos = ((datos[meteorologicas] < q1 - 1.5 * iqr) |
            (datos[meteorologicas] > q3 + 1.5 * iqr)).sum()
display(extremos.rename("posibles_extremos_IQR").to_frame())


La energia estimada presenta valores nulos en los bloques donde faltan lecturas meteorologicas.
Los ceros nocturnos son esperables en este escenario; los valores positivos con `is_day=0` o los
ceros con `is_day=1` se revisan como controles de coherencia.

La precipitacion tiene los cuartiles a cero. No eliminamos sus valores extremos por ello, ni
eliminamos valores poco frecuentes de energia, presion, viento o visibilidad sin comprobar su
origen.

## 3. Revisión del cruce de datos solares
Antes de interpretar relaciones, comprobamos cuántas mediciones solares distintas hay realmente.


In [ ]:
display(datos["radiation_quality"].value_counts().rename("filas").to_frame())

grupos_origen = datos.groupby(["station_id", "radiation_source_timestamp"]).agg(
    filas=(objetivo, "size"),
    valores_objetivo=(objetivo, "nunique")
)
display(grupos_origen.value_counts().rename("grupos").to_frame())

datos["minutos_desde_origen"] = (
    datos["timestamp"] - datos["radiation_source_timestamp"]
).dt.total_seconds() / 60
display(datos["minutos_desde_origen"].describe().to_frame())

# Una observación por estación y hora de origen para las comparaciones descriptivas.
# La meteorología se resume con su media dentro de esa hora.
agregacion = {variable: "mean" for variable in meteorologicas}
agregacion.update({variable: "first" for variable in solares})
agregacion.update({"station_name": "first", "is_day": "first"})
datos_hora = datos.groupby(["station_id", "radiation_source_timestamp"], as_index=False).agg(agregacion)
datos_hora["fecha_local"] = datos_hora["radiation_source_timestamp"].dt.tz_convert(zona_aruba)
datos_hora["hora"] = datos_hora["fecha_local"].dt.hour
datos_hora["mes"] = datos_hora["fecha_local"].dt.strftime("%Y-%m")
datos_hora_dia = datos_hora[datos_hora["is_day"] == 1].copy()

print("Filas de cinco minutos:", len(datos))
print("Grupos estación-hora de origen:", len(datos_hora))
print("Grupos diurnos:", len(datos_hora_dia))


Todas las filas llevan la etiqueta `repetido_desde_hora`. Hay **18.332 grupos estación-hora**
y cada grupo contiene doce filas con el mismo valor de radiación. La rejilla de cinco minutos
no aporta doce mediciones solares independientes por hora.

Usaremos la serie de cinco minutos para observar los escalones y preparar los retardos.
Para las principales relaciones con la meteorología trabajaremos por hora de origen y
separaremos el periodo diurno. Las medias meteorológicas horarias son solo un resumen para
este EDA: no deben utilizarse para predecir desde el principio de una hora, porque incluirían
lecturas posteriores al instante de predicción.

La fecha de origen identifica el periodo del dato, pero no demuestra cuándo estaba disponible
en tiempo real. Esa disponibilidad tendrá que comprobarse antes del modelado.


## 4. Distribucion de la energia estimada y de las variables meteorologicas
Separamos dia y noche para que la acumulacion de ceros no oculte la variabilidad de la energia.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=datos_hora, x=objetivo, bins=40, ax=ejes[0], color="steelblue")
ejes[0].set_title("Energía estimada: todas las horas")
sns.histplot(data=datos_hora_dia, x=objetivo, bins=40, ax=ejes[1], color="darkorange")
ejes[1].set_title("Energía estimada: solo horas diurnas")
for eje in ejes:
    eje.set_xlabel("Energía estimada (W/m²)")
    eje.set_ylabel("Grupos estación-hora")
plt.tight_layout()
plt.show()

display(datos_hora.groupby("is_day")[objetivo].agg(["count", "mean", "median", "min", "max"]).round(2))


In [ ]:
variables_distribucion = ["temperature_c", "humidity_pct", "wind_speed_kmh", "pressure_hpa", "uv_index", "cloud_cover"]
etiquetas = {
    "temperature_c": "Temperatura (°C)",
    "humidity_pct": "Humedad (%)",
    "wind_speed_kmh": "Velocidad del viento (km/h)",
    "wind_direction_deg": "Dirección del viento (grados)",
    "pressure_hpa": "Presión (hPa)",
    "precipitation_mm": "Precipitación (mm)",
    "visibility_km": "Visibilidad (km)",
    "uv_index": "Índice UV",
    "cloud_cover": "Nubosidad (%)"
}

fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
for variable, eje in zip(variables_distribucion, ejes.flat):
    sns.histplot(data=datos_hora, x=variable, bins=30, ax=eje, color="steelblue")
    eje.set_title("Distribución de " + etiquetas[variable].lower())
    eje.set_xlabel(etiquetas[variable])
    eje.set_ylabel("Grupos estación-hora")
plt.tight_layout()
plt.show()


Los ceros tienen mucho peso en la distribucion de la energia estimada al incluir las horas
nocturnas. Comparamos las correlaciones globales con las diurnas para no confundir el ciclo
dia/noche con relaciones meteorologicas.

## 5. Evolucion temporal y diferencias entre estaciones
Revisamos el perfil horario, la evolucion diaria y las diferencias por mes y estacion de la
energia estimada. Las magnitudes siguen representando un escenario calculado, no produccion medida.

In [ ]:
perfil = datos_hora.groupby(["hora", "station_name"])[objetivo].mean().unstack("station_name")
perfil.plot(marker="o")
plt.title("Energía estimada media por hora local y estación")
plt.xlabel("Hora de Aruba (UTC-4)")
plt.ylabel("Energía estimada media (W/m²)")
plt.xticks(range(0, 24, 2))
plt.legend(title="Estación")
plt.tight_layout()
plt.show()

# Mostramos una semana para apreciar la repetición horaria en la rejilla de cinco minutos.
semana = datos[(datos["fecha_local"] >= "2026-03-02") &
               (datos["fecha_local"] < "2026-03-09")]
for estacion, grupo in semana.groupby("station_name"):
    plt.plot(grupo["fecha_local"], grupo[objetivo], label=estacion, linewidth=1)
plt.title("Radiación cada cinco minutos durante una semana de marzo")
plt.xlabel("Fecha y hora local")
plt.ylabel("Energía estimada (W/m²)")
plt.legend(title="Estación")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Excluimos los días locales incompletos de la media diaria.
diario = datos.groupby(["station_name", pd.Grouper(key="fecha_local", freq="D")]).agg(
    energia_media=(objetivo, "mean"),
    filas=(objetivo, "size")
).reset_index()
diario_completo = diario[diario["filas"] == 288]

for estacion, grupo in diario_completo.groupby("station_name"):
    plt.plot(grupo["fecha_local"], grupo["energia_media"], label=estacion, linewidth=1, alpha=0.8)
plt.title("Evolución de la energía estimada media diaria en días completos")
plt.xlabel("Fecha local")
plt.ylabel("Energía estimada media de las 24 horas (W/m²)")
plt.legend(title="Estación")
plt.tight_layout()
plt.show()

mensual = datos_hora_dia.groupby(["mes", "station_name"])[objetivo].agg(["count", "mean"]).round(2)
display(mensual)
mensual["mean"].unstack("station_name").plot(marker="o")
plt.title("Energía estimada media diurna por mes y estación")
plt.xlabel("Mes local (primer y último mes pueden ser parciales)")
plt.ylabel("Energía estimada media diurna (W/m²)")
plt.xticks(rotation=45)
plt.legend(title="Estación")
plt.tight_layout()
plt.show()


In [ ]:
display(datos_hora_dia.groupby("station_name")[objetivo].agg(["count", "mean", "median", "std", "max"]).round(2))
sns.boxplot(data=datos_hora_dia, x="station_name", y=objetivo, color="lightblue")
plt.title("Distribución de la energía estimada diurna por estación")
plt.xlabel("Estación")
plt.ylabel("Energía estimada (W/m²)")
plt.tight_layout()
plt.show()


Las medias de radiación de las dos estaciones son muy próximas: en las filas originales
son 262,17 y 262,19 W/m² al incluir noche y día. El máximo del perfil medio aparece alrededor
de las 13:00 locales. La forma escalonada de la serie refleja la repetición de datos horarios.

Hay solo dos ubicaciones. No podemos separar con estos datos un efecto de la altitud o de
las coordenadas del efecto propio de cada estación. Además, las diferencias entre ambas
pueden depender de cómo se obtuvo la radiación y no solo del microclima.


## 6. Relaciones de la meteorologia con la energia estimada
Comenzamos con correlaciones y despues vemos las relaciones una a una. Pearson resume relaciones
lineales y Spearman compara el orden de los valores. Ninguna demuestra causalidad.

In [ ]:
predictoras = ["temperature_c", "humidity_pct", "wind_speed_kmh", "pressure_hpa",
               "precipitation_mm", "visibility_km", "uv_index", "cloud_cover"]

relaciones = pd.DataFrame({
    "Pearson_5min_todas": datos[predictoras].corrwith(datos[objetivo]),
    "Pearson_hora_todas": datos_hora[predictoras].corrwith(datos_hora[objetivo]),
    "Pearson_hora_dia": datos_hora_dia[predictoras].corrwith(datos_hora_dia[objetivo]),
    "Spearman_hora_dia": datos_hora_dia[predictoras + [objetivo]].corr(method="spearman")[objetivo].reindex(predictoras),
    "parejas_hora_dia": datos_hora_dia[predictoras].notna().sum()
})
display(relaciones.round(3))

relaciones[["Pearson_hora_todas", "Pearson_hora_dia"]].rename(index=etiquetas).plot.barh()
plt.title("Relación con la radiación: todas las horas frente a horas diurnas")
plt.xlabel("Correlación de Pearson")
plt.ylabel("Variable")
plt.xlim(-1, 1)
plt.axvline(0, color="black", linewidth=0.8)
plt.legend(["Todas las horas", "Solo horas diurnas"])
plt.tight_layout()
plt.show()


In [ ]:
matriz = datos_hora_dia[predictoras + [objetivo]].corr()
matriz = matriz.rename(index=etiquetas, columns=etiquetas)
matriz = matriz.rename(index={objetivo: "Energía estimada"}, columns={objetivo: "Energía estimada"})
plt.figure(figsize=(12, 9))
sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlaciones entre meteorología y radiación en horas diurnas")
plt.tight_layout()
plt.show()


In [ ]:
# Usamos todos los grupos horarios diurnos; alpha ayuda a ver dónde se concentran.
fig, ejes = plt.subplots(4, 2, figsize=(14, 17))
for variable, eje in zip(predictoras, ejes.flat):
    sns.scatterplot(data=datos_hora_dia, x=variable, y=objetivo,
                    hue="station_name", alpha=0.18, s=10, ax=eje)
    eje.set_title("Radiación frente a " + etiquetas[variable].lower())
    eje.set_xlabel(etiquetas[variable])
    eje.set_ylabel("Energía estimada (W/m²)")
    eje.legend(title="Estación", markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()


En las filas de cinco minutos, temperatura y viento presentan correlaciones negativas con
la radiación (aproximadamente -0,55 y -0,65), mientras que la humedad presenta una correlación
positiva (0,50). Al considerar solo el día, estas asociaciones se reducen.

Estos signos no bastan para afirmar que una temperatura más baja produzca más radiación o
que una mayor humedad la favorezca. Pueden intervenir la hora, el cruce de fuentes y el
periodo del año. El índice UV también muestra una relación llamativamente débil en el
conjunto completo. Revisamos los perfiles horarios antes de sacar conclusiones.

La dirección del viento no se incluye en la correlación lineal principal porque es una
variable circular: 0 y 360 grados representan la misma dirección.


### 6.1 Comprobación de la alineación entre UV y radiación
Comparamos la forma de los perfiles, dividiendo cada variable por su máximo horario.
Así se ven en una escala común; el gráfico no compara sus unidades originales.


In [ ]:
perfil_comparado = datos_hora.groupby("hora")[[objetivo, "uv_index"]].mean()
perfil_normalizado = perfil_comparado / perfil_comparado.max()
perfil_normalizado.rename(columns={objetivo: "Energía estimada", "uv_index": "Índice UV"}).plot(marker="o")
plt.title("Comparación de los perfiles horarios de radiación y UV")
plt.xlabel("Hora de Aruba (UTC-4)")
plt.ylabel("Media horaria / máximo del perfil")
plt.xticks(range(0, 24, 2))
plt.legend(title="Variable")
plt.tight_layout()
plt.show()

print("Hora local del máximo medio de radiación:", perfil_comparado[objetivo].idxmax())
print("Hora local del máximo medio de UV:", perfil_comparado["uv_index"].idxmax())
print("Filas con UV positivo y is_day=0:", ((datos["uv_index"] > 0) & (datos["is_day"] == 0)).sum())


El máximo medio de UV aparece a las **12:00 locales** y el de radiación a las **13:00**.

### 6.2 Nubosidad y radiación a horas comparables
La hora modifica mucho la radiación. Comparamos bandas de nubosidad dentro de cada hora
para no mezclar, por ejemplo, una mañana nublada con un mediodía despejado.


In [ ]:
datos_hora_dia["grupo_nubes"] = pd.cut(
    datos_hora_dia["cloud_cover"], bins=[-0.1, 25, 50, 75, 100],
    labels=["0-25 %", "25-50 %", "50-75 %", "75-100 %"]
)
tabla_nubes = datos_hora_dia.pivot_table(index="hora", columns="grupo_nubes",
                                      values=objetivo, aggfunc="mean", observed=True)
recuento_nubes = datos_hora_dia.pivot_table(index="hora", columns="grupo_nubes",
                                         values=objetivo, aggfunc="count", observed=True)

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(tabla_nubes, cmap="YlOrRd", annot=True, fmt=".0f", ax=ejes[0],
            cbar_kws={"label": "Energía estimada media (W/m²)"})
ejes[0].set_title("Energía estimada media según hora local y nubosidad")
sns.heatmap(recuento_nubes, cmap="Blues", annot=True, fmt=".0f", ax=ejes[1],
            cbar_kws={"label": "Grupos estación-hora"})
ejes[1].set_title("Número de observaciones por hora y nubosidad")
for eje in ejes:
    eje.set_xlabel("Nubosidad")
    eje.set_ylabel("Hora local")
plt.tight_layout()
plt.show()


El segundo gráfico permite comprobar cuánto respaldo tiene cada media. Las celdas con pocas
observaciones deben interpretarse con cuidado. Si la relación con nubosidad sigue siendo
débil a una misma hora, habrá que revisar la fuente y la correspondencia del cruce.


### 6.3 Lluvia, visibilidad y viento
La lluvia puede estar relacionada con las condiciones del cielo y también con la limpieza
de paneles. Aquí solo podemos estudiar su relación con la radiación, porque no tenemos
mediciones de suciedad ni producción de los paneles.


In [ ]:
lluvia = datos_hora_dia.dropna(subset=["precipitation_mm"]).copy()
lluvia["situacion"] = "Sin lluvia"
lluvia.loc[lluvia["precipitation_mm"] > 0, "situacion"] = "Con lluvia"
display(lluvia.groupby("situacion")[objetivo].agg(["count", "mean", "median"]).round(2))

sns.boxplot(data=lluvia, x="situacion", y=objetivo, order=["Sin lluvia", "Con lluvia"], color="lightblue")
plt.title("Energía estimada diurna en horas con y sin lluvia registrada")
plt.xlabel("Situación según la precipitación media horaria")
plt.ylabel("Energía estimada (W/m²)")
plt.tight_layout()
plt.show()

# La dirección se muestra en su escala original, sin tratarla como correlación lineal.
sns.scatterplot(data=datos_hora_dia, x="wind_direction_deg", y=objetivo,
                alpha=0.2, s=12, color="steelblue")
plt.title("Energía estimada diurna y dirección media del viento")
plt.xlabel("Dirección media del viento (grados)")
plt.ylabel("Energía estimada (W/m²)")
plt.tight_layout()
plt.show()


La mayoría de registros no tiene lluvia y la visibilidad varía poco en buena parte de la serie.
Una correlación global pequeña no descarta diferencias en episodios concretos. La comparación
con/sin lluvia también puede reflejar diferencias de hora o estación. No demuestra un efecto
de lavado de los paneles. La dirección del viento del archivo se concentra entre unos 18 y 73
grados; dentro de ese rango su media aritmética sirve como resumen descriptivo.


## 7. Relación con las otras variables solares
Estas variables comparten información física con la radiación total. Una relación alta puede
ser esperable por su definición y no indicar una variable independiente útil para predecir.


In [ ]:
componentes = ["direct_radiation_instant", "diffuse_radiation_instant", "direct_normal_irradiance_instant"]
nombres_componentes = ["Radiación directa", "Radiación difusa", "Irradiancia normal directa"]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for variable, nombre, eje in zip(componentes, nombres_componentes, ejes):
    sns.scatterplot(data=datos_hora_dia, x=variable, y=objetivo, alpha=0.15, s=10, ax=eje)
    eje.set_title("Energía estimada frente a " + nombre.lower())
    eje.set_xlabel(nombre + " (W/m²)")
    eje.set_ylabel("Energía estimada (W/m²)")
plt.tight_layout()
plt.show()

display(datos_hora_dia[[objetivo] + componentes + ["sunshine_duration_source_s"]]
        .corr()[[objetivo]].round(3))

diferencia_componentes = datos_hora[objetivo] - (
    datos_hora["direct_radiation_instant"] + datos_hora["diffuse_radiation_instant"]
)
display(diferencia_componentes.describe().rename("total_menos_directa_y_difusa").to_frame())


La radiación directa y la difusa permiten comprobar la coherencia de la radiación total.
La irradiancia normal directa tiene otra referencia geométrica y no se suma directamente
a la difusa para obtener el total horizontal.

Si usáramos las componentes del mismo instante para explicar el total de ese instante,
estaríamos reconstruyendo una magnitud muy relacionada por definición. Para una predicción
futura solo podríamos usar los valores que estuvieran disponibles en el momento de predecir.
La duración de insolación del periodo de origen también requiere revisar cuándo se conoce.


## 8. Retardos y energia estimada a 15 minutos
El objetivo futuro es `energia_estimada_kwh` y se preparan retardos de 5, 10, 15 y 20 minutos.
Los desplazamientos se hacen dentro de cada estacion y se comprueba la distancia temporal para
no mezclar estaciones ni aceptar huecos como si fueran quince minutos.

In [ ]:
base = datos.copy()
lags = []
for minutos in [5, 10, 15, 20]:
    pasos = minutos // 5
    nombre = "energia_lag_" + str(minutos) + "min"
    fecha_anterior = base.groupby("station_id")["timestamp"].shift(pasos)
    valor_anterior = base.groupby("station_id")[objetivo].shift(pasos)
    intervalo_correcto = base["timestamp"] - fecha_anterior == pd.Timedelta(minutes=minutos)
    base[nombre] = valor_anterior.where(intervalo_correcto)
    lags.append(nombre)

base["tiempo_objetivo"] = base.groupby("station_id")["timestamp"].shift(-3)
base["energia_t15"] = base.groupby("station_id")[objetivo].shift(-3)
intervalo_futuro = base["tiempo_objetivo"] - base["timestamp"] == pd.Timedelta(minutes=15)
base["energia_t15"] = base["energia_t15"].where(intervalo_futuro)

# Estas dos columnas son solo para diagnosticar el origen de la etiqueta futura.
base["origen_t15"] = base.groupby("station_id")["radiation_source_timestamp"].shift(-3)
base["mismo_origen_t15"] = base["radiation_source_timestamp"] == base["origen_t15"]

display(base[["station_name", "timestamp", objetivo] + lags + ["tiempo_objetivo", "energia_t15"]].head(10))
display(base[lags + ["energia_t15"]].isna().sum().rename("ausentes_por_desplazamiento").to_frame())


In [ ]:
relaciones_futuras = pd.DataFrame({
    "con_energia_actual": base[predictoras + lags].corrwith(base[objetivo]),
    "con_energia_t15": base[predictoras + lags].corrwith(base["energia_t15"])
})
base_dia = base[base["is_day"] == 1]
relaciones_futuras["con_energia_t15_solo_dia"] = base_dia[predictoras + lags].corrwith(base_dia["energia_t15"])
display(relaciones_futuras.round(3))

base_dia[lags + ["energia_t15"]].corr()["energia_t15"].drop("energia_t15").plot.bar(color="steelblue")
plt.title("Relación de los retardos con la radiación a 15 minutos en periodo diurno")
plt.xlabel("Retardo de radiación")
plt.ylabel("Correlación de Pearson")
plt.ylim(-1, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

comparacion = base.dropna(subset=[objetivo, "energia_t15"]).copy()
comparacion["sin_cambio"] = comparacion[objetivo] == comparacion["energia_t15"]
display(comparacion.groupby("mismo_origen_t15").agg(
    filas=("energia_t15", "size"),
    proporcion_sin_cambio=("sin_cambio", "mean")
).round(3))
print("Porcentaje total sin cambio a 15 minutos:", round(comparacion["sin_cambio"].mean() * 100, 2))
print("Porcentaje diurno sin cambio:", round(
    comparacion.loc[comparacion["is_day"] == 1, "sin_cambio"].mean() * 100, 2
))


En una parte importante de las parejas, la energia estimada actual y la de quince minutos
despues son iguales. Esto incluye ceros nocturnos y valores repetidos dentro de la misma hora.

Para evaluar un modelo sera necesario comparar con una referencia sencilla: mantener la energia
estimada actual como prediccion. Aqui no entrenamos un modelo ni presentamos metricas predictivas.

## 9. Preparación de una separación temporal
Dejamos planteada la división cronológica 80/20 del anteproyecto. Las dos estaciones comparten
el mismo corte y no repartimos una hora de origen entre entrenamiento y prueba. También
retiramos del entrenamiento las filas cuya etiqueta a 15 minutos llegue al periodo de prueba.


In [ ]:
# Elegimos el corte sobre las horas de origen antes de quitar filas por ausentes.
horas_origen = base["radiation_source_timestamp"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
corte = horas_origen.iloc[int(len(horas_origen) * 0.8)]

# Selección provisional: no incluye variables futuras ni las medias horarias del EDA.
entradas = predictoras + [objetivo, "hora"] + lags
base_modelo = base.dropna(subset=entradas + ["energia_t15", "tiempo_objetivo"]).copy()

entrenamiento = base_modelo[(base_modelo["radiation_source_timestamp"] < corte) &
                            (base_modelo["tiempo_objetivo"] < corte)].copy()
prueba = base_modelo[base_modelo["radiation_source_timestamp"] >= corte].copy()

print("Corte temporal (UTC):", corte)
print("Filas antes de quitar ausentes:", len(base))
print("Filas con entradas y objetivo completos:", len(base_modelo))
print("Filas de entrenamiento:", len(entrenamiento))
print("Filas de prueba:", len(prueba))
print("Filas retiradas por cruce de la etiqueta con el corte:",
      len(base_modelo) - len(entrenamiento) - len(prueba))
print("Último instante objetivo de entrenamiento:", entrenamiento["tiempo_objetivo"].max())
print("Primer instante de prueba:", prueba["timestamp"].min())


La proporción final de filas puede diferir de 80/20 por los ausentes y por las filas retiradas
en el límite. Las lecturas anteriores al corte sí pueden servir de historial para los primeros
retardos de prueba, porque ya pertenecen al pasado en el instante de predecir.

La selección de variables es provisional: antes de entrenar hay que resolver la alineación
del UV y comprobar la disponibilidad temporal de todas las fuentes. El EDA usa el periodo
completo para describirlo; la selección definitiva, los ajustes y cualquier imputación del
modelo deberán decidirse usando solo entrenamiento. Las métricas RMSE y R² se calcularán
cuando exista un modelo y un objetivo adecuado, como indica el anteproyecto.


## 10. Conclusiones y relación con el anteproyecto

1. **Disponibilidad y calidad.** Tenemos dos estaciones con una rejilla regular de cinco minutos,
   sin duplicados de estación-fecha. Hay 8.640 filas sin meteorología que no hemos rellenado.
   La preparación responde a RF-01 y conserva la identificación de las estaciones de RF-02.
2. **Resolución real de la radiación.** Los doce registros de cada estación-hora comparten el mismo
   valor solar. No deben interpretarse como doce mediciones independientes. Esto limita lo
   que podremos evaluar a horizontes de 5 a 20 minutos.
3. **Patrones temporales.** La radiación tiene un ciclo diario muy marcado y las medias de las
   estaciones son muy próximas. Las diferencias mensuales describen este periodo, sin demostrar
   una estacionalidad estable a largo plazo.
4. **Relaciones con las variables.** Hemos comparado temperatura, humedad, viento, presión,
   lluvia, visibilidad, UV y nubosidad con la radiación. Separar las horas diurnas cambia las
   correlaciones. Las componentes solares comparten información por su propia definición.
5. **Preparación temporal.** Los retardos de 5, 10, 15 y 20 minutos se construyen por estación,
   con comprobación del intervalo. La etiqueta provisional es radiación a 15 minutos y dejamos
   una división cronológica sin etiquetas de entrenamiento que alcancen el periodo de prueba.
   Esto prepara RF-07 y RF-10, pero no sustituye la evaluación del modelo.
6. **Lo que falta para el objetivo final.** Necesitamos generación o potencia medida con fechas
   compatibles y datos de las instalaciones y del estado de limpieza de los paneles.
   Sin ellos no podemos cuantificar el efecto de la suciedad ni predecir producción eléctrica
   real. Tampoco podemos fijar un umbral de déficit energético usando solo radiación.

El siguiente paso es revisar el cruce temporal y la fuente solar. Una vez resuelto, podremos
incorporar los datos de generación y de paneles y preparar el modelo del anteproyecto.

**Fuentes:** `Solar Aruba Anteproyecto (1).pdf`, apartados 1.2-1.4, RF-01, RF-02, RF-07,
RF-09, RF-10, RF-13 y objetivos del apartado 3; `aruba_cruzado_5min (1).csv`.
